In [1]:
"""
Day 6: PyTorch-idiomatic refactor + AMP training
-------------------------------------------------
Re-implements the Day 5 character-level MLP using idiomatic PyTorch:
nn.Module, Dataset/DataLoader, torch.optim, .to(device), and AMP.

Same architecture and math as Day 5 -- every hand-rolled primitive is
swapped for its production counterpart:

    C = randn + C[X]            ->  nn.Embedding
    W1,b1 + emb @ W1 + b1       ->  nn.Linear(bias=False)   (bias killed by BN)
    bngain/bnbias + running     ->  nn.BatchNorm1d
    W2,b2 + h @ W2 + b2         ->  nn.Linear
    parameters=[...]            ->  model.parameters()
    p.grad = None               ->  optimizer.zero_grad()
    p.data += -lr * p.grad      ->  optimizer.step()  (AdamW)
    torch.randint minibatch     ->  DataLoader(shuffle=True)
    --                          ->  .to(device)
    --                          ->  torch.autocast + GradScaler

Requires names.txt in the working directory (Karpathy makemore dataset).
The `# %%` markers let this run as VSCode/Jupyter cells matching the
five-block build, or straight as a script.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random


# %% Block 0: data ----------------------------------------------------------
words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                                    # 0 reserved for start/end token
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)                           # 27

block_size = 3                                   # context length

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size               # start with three '.'
        for ch in w + '.':                       # trailing '.' teaches name-end
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]         # roll the window forward
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)                                  # reproducible split
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr,  Ytr  = build_dataset(words[:n1])           # 80%
Xdev, Ydev = build_dataset(words[n1:n2])         # 10%
Xte,  Yte  = build_dataset(words[n2:])           # 10%

print(f"vocab_size: {vocab_size}")
print(f"Xtr: {Xtr.shape}, Ytr: {Ytr.shape}")


# %% Block 1: dataset + loader ----------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

class NamesDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_ds = NamesDataset(Xtr, Ytr)
dev_ds   = NamesDataset(Xdev, Ydev)

# drop_last=True drops the final incomplete batch. len(Xtr) % 32 == 1, so the
# tail batch is a singleton -- and BatchNorm1d cannot compute a batch variance
# over one example in train() mode (same single-example failure as Day 5).
# dev_loader needs no drop_last: eval() uses running stats, so size-1 is fine,
# and dropping dev examples would skew the loss.
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
dev_loader   = DataLoader(dev_ds, batch_size=256, shuffle=False)

print(f"train examples: {len(train_ds)}")
Xb, Yb = next(iter(train_loader))
print(f"batch X: {Xb.shape}, batch Y: {Yb.shape}")


# %% Block 2: model ---------------------------------------------------------
n_emb    = 10
n_hidden = 200

class MLP(nn.Module):
    def __init__(self, vocab_size, block_size, n_emb, n_hidden):
        super().__init__()                                       # registers submodules
        self.embedding = nn.Embedding(vocab_size, n_emb)         # was C[X]
        self.fc1 = nn.Linear(block_size * n_emb, n_hidden, bias=False)  # bias killed by BN
        self.bn1 = nn.BatchNorm1d(n_hidden)                      # absorbs all hand-rolled BN
        self.fc2 = nn.Linear(n_hidden, vocab_size)              # keeps bias (no BN after)

    def forward(self, x):                        # x: (B, block_size) int indices
        emb = self.embedding(x)                  # (B, block_size, n_emb)
        emb = emb.view(emb.shape[0], -1)         # (B, block_size * n_emb)
        h = self.fc1(emb)                        # (B, n_hidden)
        h = self.bn1(h)                          # (B, n_hidden)
        h = torch.tanh(h)                        # (B, n_hidden)
        return self.fc2(h)                       # (B, vocab_size) raw logits

torch.manual_seed(42)
model = MLP(vocab_size, block_size, n_emb, n_hidden).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameters: {n_params}")                 # 12097


# %% Block 3: sanity (step-0 loss) ------------------------------------------
Xb, Yb = next(iter(train_loader))
Xb, Yb = Xb.to(device), Yb.to(device)

logits = model(Xb)                               # routes through __call__ -> forward
loss = F.cross_entropy(logits, Yb)               # log-softmax + NLL, stable

print(f"logits shape: {logits.shape}")           # (32, 27)
print(f"step-0 loss:  {loss.item():.4f}")        # ~3.3  (ln(27) = 3.2958)


# %% Block 4: AMP training loop ---------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

use_amp = (device == 'cuda')                      # fp16 AMP is a CUDA path
scaler = torch.amp.GradScaler(device, enabled=use_amp)  # scales grads out of fp16 underflow

epochs = 5
model.train()                                    # BN -> batch stats + accumulate running

for epoch in range(epochs):
    running = 0.0
    for step, (Xb, Yb) in enumerate(train_loader):
        Xb, Yb = Xb.to(device), Yb.to(device)

        optimizer.zero_grad()                    # was: for p: p.grad = None

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(Xb)                   # fast ops in fp16, reductions in fp32
            loss = F.cross_entropy(logits, Yb)   # only forward + loss go inside autocast

        scaler.scale(loss).backward()            # scale up -> backprop (no underflow)
        scaler.step(optimizer)                   # unscale -> check inf/NaN -> step
        scaler.update()                          # adapt scale factor for next iter

        running += loss.item()                   # .item() = true (unscaled) loss
        if step % 1000 == 0:
            print(f"  epoch {epoch+1} step {step:5d}  loss {loss.item():.4f}")

    print(f"epoch {epoch+1}/{epochs}  avg loss {running/len(train_loader):.4f}")


# %% Block 5: eval + sampling -----------------------------------------------
@torch.no_grad()                                 # no grad tracking for the whole fn
def evaluate(loader):
    model.eval()                                 # BN -> running stats (the Day 5 payoff)
    total_loss, n = 0.0, 0
    for Xb, Yb in loader:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb, reduction='sum')  # sum, then divide by true n
        total_loss += loss.item()
        n += Yb.shape[0]
    return total_loss / n

print(f"train loss: {evaluate(train_loader):.4f}")
print(f"dev   loss: {evaluate(dev_loader):.4f}")

# ---- sample names ----
model.eval()                                     # batch-size-1 forwards need running stats
g = torch.Generator(device=device).manual_seed(2147483647)

for _ in range(20):
    out = []
    context = [0] * block_size                   # start with all '.'
    while True:
        x = torch.tensor([context], device=device)        # (1, block_size)
        with torch.no_grad():
            logits = model(x)                              # (1, 27)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()  # weighted draw
        context = context[1:] + [ix]             # roll the window
        out.append(ix)
        if ix == 0:                              # hit '.' end token
            break
    print(''.join(itos[i] for i in out))

vocab_size: 27
Xtr: torch.Size([182625, 3]), Ytr: torch.Size([182625])
device: cuda
train examples: 182625
batch X: torch.Size([32, 3]), batch Y: torch.Size([32])
MLP(
  (embedding): Embedding(27, 10)
  (fc1): Linear(in_features=30, out_features=200, bias=False)
  (bn1): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=200, out_features=27, bias=True)
)
parameters: 12097
logits shape: torch.Size([32, 27])
step-0 loss:  3.3596
  epoch 1 step     0  loss 3.3582
  epoch 1 step  1000  loss 2.1767
  epoch 1 step  2000  loss 2.5013
  epoch 1 step  3000  loss 2.1855
  epoch 1 step  4000  loss 2.2105
  epoch 1 step  5000  loss 2.3025
epoch 1/5  avg loss 2.3630
  epoch 2 step     0  loss 2.3201
  epoch 2 step  1000  loss 2.0457
  epoch 2 step  2000  loss 2.3288
  epoch 2 step  3000  loss 2.2659
  epoch 2 step  4000  loss 2.0823
  epoch 2 step  5000  loss 2.7863
epoch 2/5  avg loss 2.2561
  epoch 3 step     0  loss 2.1552
  epoch 3 step